# Onset-conditional evaluation (added after peer review)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# =====================================================================
# ONSET-CONDITIONAL EVALUATION (added after peer review)
#
# Question: on days with NO MHW on day t, can the model warn that an MHW
# will be present on day t+lead? Persistence always says "no" on these
# days, so it has no skill here by definition (ROC-AUC 0.5, recall 0).
#
# Uses only saved files (no models, no raw data):
#   OUTPUT_DIR/test_predictions/{model}_{config}_lead{lead}.parquet
#   OUTPUT_DIR/best_per_model.csv   (variant + threshold chosen on THR split)
#
# For each model x config x lead, on the subset y_now == 0:
#   n_days, n_onsets, onset_rate           subset size and event rate
#   auc                                     ranking skill (0.5 = none)
#   sedi, tpr (onset recall), fpr           at the SAME threshold as Table II
#   precision                               share of onset warnings that came true
#   bss                                     vs climatology of this subset
#   *_pers                                  persistence on the same subset
# =====================================================================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, brier_score_loss

BASE_DIR   = Path('/content/drive/MyDrive/MSc-research-code/')
OUTPUT_DIR = BASE_DIR / 'evaluation_results/'
PRED_DIR   = OUTPUT_DIR / 'test_predictions'
CONFIGS    = ['Sub-SST', '0-50m', '50-100m', '100-150m']
LEAD_TIMES = [3, 5, 7, 9, 12, 14]
MODELS     = ['LightGBM', 'LSTM']


def sedi_score(tp, fp, fn, tn):
    """Exact copy of sedi_score() in EvaluationWithCalibration (same results)."""
    eps = 1e-7
    tpr = tp / (tp + fn) if (tp + fn) > 0 else eps
    fpr = fp / (fp + tn) if (fp + tn) > 0 else eps
    H = np.clip(tpr, eps, 1 - eps)
    F = np.clip(fpr, eps, 1 - eps)
    try:
        num = np.log(F) - np.log(H) - np.log(1 - F) + np.log(1 - H)
        den = np.log(F) + np.log(H) + np.log(1 - F) + np.log(1 - H)
        val = num / den
        return 0.0 if (np.isnan(val) or np.isinf(val)) else float(val)
    except Exception:
        return 0.0


def bss_vs_subset_climatology(y, p):
    r = float(y.mean())
    return 0.0 if r * (1 - r) == 0 else float(1 - brier_score_loss(y, p) / (r * (1 - r)))


best = pd.read_csv(OUTPUT_DIR / 'best_per_model.csv')
assert {'brier_thr', 'ece_thr'} <= set(best.columns), \
    'use best_per_model.csv from the v5 run (THR-split selection)'

rows = []
for model in MODELS:
    for config in CONFIGS:
        for lead in LEAD_TIMES:
            f = PRED_DIR / f'{model}_{config}_lead{lead}.parquet'
            sel = best[(best['model'] == model) & (best['config'] == config)
                       & (best['lead_time'] == lead)]
            if not f.exists() or sel.empty:
                print(f'  missing: {model} {config} lead {lead}')
                continue
            sel = sel.iloc[0]
            pcol, thr = f"p__{sel['calibration']}", float(sel['best_threshold'])

            d = pd.read_parquet(f, columns=['y', 'y_now', pcol])
            d = d[d['y_now'] == 0]                       # no MHW on day t
            y = d['y'].values.astype(np.int8)            # MHW on day t+lead = onset
            p = d[pcol].values.astype(np.float64)
            pred = (p > thr).astype(int)                 # '>' as in main evaluation

            tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
            fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
            has_both = 0 < y.sum() < len(y)

            rows.append({
                'model': model, 'config': config, 'lead_time': lead,
                'calibration': sel['calibration'], 'threshold': thr,
                'n_days': len(y), 'n_onsets': int(y.sum()),
                'onset_rate': float(y.mean()),
                'auc': float(roc_auc_score(y, p)) if has_both else np.nan,
                'sedi': sedi_score(tp, fp, fn, tn),
                'tpr': tp / (tp + fn) if (tp + fn) else np.nan,      # onset recall
                'fpr': fp / (fp + tn) if (fp + tn) else np.nan,      # false alarm rate
                'precision': tp / (tp + fp) if (tp + fp) else np.nan,
                'bss': bss_vs_subset_climatology(y, p),
                # persistence forecasts "no MHW" on every one of these days
                'auc_pers': 0.5, 'tpr_pers': 0.0, 'fpr_pers': 0.0,
                'bss_pers': bss_vs_subset_climatology(y, np.zeros(len(y))),
            })
    print(f'{model}: done')

onset_df = pd.DataFrame(rows)
onset_df.to_csv(OUTPUT_DIR / 'onset_conditional.csv', index=False)
print(f'\nSaved: onset_conditional.csv ({len(onset_df)} rows, expected '
      f'{len(MODELS) * len(CONFIGS) * len(LEAD_TIMES)})')
print(onset_df[['model', 'config', 'lead_time', 'n_onsets', 'onset_rate',
                'auc', 'sedi', 'tpr', 'fpr', 'precision', 'bss']]
      .sort_values(['config', 'model', 'lead_time']).round(3).to_string(index=False))


LightGBM: done
LSTM: done

Saved: onset_conditional.csv (48 rows, expected 48)
   model   config  lead_time  n_onsets  onset_rate   auc  sedi   tpr   fpr  precision    bss
    LSTM    0-50m          3     43231       0.092 0.644 0.245 0.488 0.318      0.134 -0.220
    LSTM    0-50m          5     65752       0.140 0.655 0.265 0.524 0.338      0.201 -0.062
    LSTM    0-50m          7     81216       0.173 0.647 0.262 0.590 0.405      0.233 -0.004
    LSTM    0-50m          9     92128       0.196 0.636 0.249 0.649 0.475      0.250  0.002
    LSTM    0-50m         12    104278       0.222 0.641 0.264 0.668 0.483      0.282  0.011
    LSTM    0-50m         14    110847       0.236 0.630 0.238 0.669 0.504      0.290 -0.002
LightGBM    0-50m          3     44080       0.092 0.634 0.229 0.468 0.310      0.133 -0.330
LightGBM    0-50m          5     66961       0.140 0.639 0.246 0.518 0.347      0.196 -0.093
LightGBM    0-50m          7     82630       0.173 0.637 0.244 0.550 0.379      0.23